# **Day 100: Writing and Publishing a Remote-Sensing ML Paper (and Course Wrap-Up)**
*Module 13 - Remote Sensing ML and Publication Projects*

We have the results: seven projects, each with tables, figures and saved outputs. Today we turn results into a **manuscript** and take it through **submission, review and publication**. Then we look back at the 100 days.

> A good paper turns results into a clear story: what was done, what was found and how certain it is. This notebook covers structure, writing from saved results, choosing a journal, responding to reviewers and publication ethics, and closes the 100 days.
>
> *Example:* "Overall accuracy was 0.95 (95% CI 0.93-0.96)" is stronger and more honest than "the model achieved excellent accuracy of 95%".

In [1]:
import numpy as np
import pandas as pd
import json, re, textwrap
from pathlib import Path
from IPython.display import Markdown, display

def load_csv(path, **kw):
    p = Path(path)
    if not p.exists(): print(f"(missing {p}: run the corresponding project notebook first)"); return None
    return pd.read_csv(p, **kw)

def md_table(df, fmt=".3f", index=True):
    """Markdown table without extra dependencies."""
    cols = ([df.index.name or ""] if index else []) + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        cells = ([str(idx)] if index else []) + [format(v, fmt) if isinstance(v, (float, np.floating)) and np.isfinite(v) else ("" if isinstance(v, (float, np.floating)) else str(v)) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

## **1. Anatomy of an RS-ML Paper**
| Section | Purpose | What reviewers check |
|---|---|---|
| **Title** | What, where, how, in at most ~20 words | Specific (place, sensor, method), no hype |
| **Abstract** | Context - gap - aim - methods - **numbers** - implication (200-300 words) | Key results with uncertainty |
| **1 Introduction** | Why it matters; what is known; the **gap**; objectives or research questions | Is the gap real? Are the RQs answerable? |
| **2 Study area and data** | Fig. 1, imagery (sensor, dates, processing level), reference data (how collected, by whom, when) | Is the reference data independent and adequate? |
| **3 Methods** | Features, models, tuning, **validation design**, accuracy and area estimation, interpretation | Spatial CV? Leakage? Probability sample? Baselines? |
| **4 Results** | Tables and figures in RQ order; numbers with CIs | Do the numbers support the claims? |
| **5 Discussion** | Interpretation, comparison with the literature, **limitations**, transferability, implications | Honest limitations; no overclaiming |
| **6 Conclusions** | Answers to the RQs in 1-2 paragraphs | Matches the results |
| Back matter | CRediT statement, funding, data and code availability, conflicts of interest, AI-use disclosure | Required by most journals |

**Write in this order**: figures and tables -> Methods -> Results -> Discussion -> Introduction -> Abstract -> Title. Write the abstract last, when we know what we found.

## **2. Results Text Generated from Saved Outputs (Project 1)**
Each project saved its tables to CSV/JSON. We build the Results section of Project 1 from them. When we re-run the analysis, the text updates with it.

In [2]:
P1 = Path("project1_lulc")
meta = json.loads((P1 / "models/final_model_meta.json").read_text()) if (P1 / "models/final_model_meta.json").exists() else None
t2 = load_csv(P1 / "outputs/tables/table2_model_comparison.csv", index_col=0)
t3 = load_csv(P1 / "outputs/tables/table3_ablation.csv")
t4 = load_csv(P1 / "outputs/tables/table4_accuracy_area.csv")
res_par = (P1 / "outputs/results_paragraph.txt").read_text() if (P1 / "outputs/results_paragraph.txt").exists() else ""
parts = ["# Results (auto-generated draft, Project 1)\n"]
if t2 is not None:
    best, second = t2.index[0], t2.index[1]
    parts.append(f"## 4.1 Model comparison (RQ1)\nUnder nested spatial cross-validation, {best} achieved the highest macro F1 "
                 f"({t2.loc[best, 'macro F1']:.3f}, 95% CI {t2.loc[best, 'macro F1 CI low']:.3f}-{t2.loc[best, 'macro F1 CI high']:.3f}), followed by {second} "
                 f"({t2.loc[second, 'macro F1']:.3f}, {t2.loc[second, 'macro F1 CI low']:.3f}-{t2.loc[second, 'macro F1 CI high']:.3f}) (Table 2).\n\n"
                 "**Table 2.** Nested spatial CV (5 outer folds of 2 km blocks): OA as mean and s.d. over folds; macro F1 from pooled out-of-fold predictions with bootstrap 95% CI.\n\n"
                 + md_table(t2[["OA (mean)", "OA (s.d.)", "macro F1", "macro F1 CI low", "macro F1 CI high"]]) + "\n")
if t3 is not None:
    parts.append(f"## 4.2 Contribution of data sources (RQ2)\nMacro F1 rose from {t3['macro F1 (pooled)'].iloc[0]:.3f} with monsoon Sentinel-2 bands alone "
                 f"to {t3['macro F1 (pooled)'].iloc[-1]:.3f} with all feature groups (Table 3).\n\n**Table 3.** Cumulative feature-group ablation (RF, 3 x 5 repeated spatial CV).\n\n"
                 + md_table(t3.set_index("feature set")[["n features", "macro F1 (pooled)", "OA (mean)", "OA (s.d.)", "OA diff vs full", "p vs full (corrected t)"]]) + "\n")
if t4 is not None:
    parts.append("## 4.3 Map accuracy and class areas (RQ3)\n" + res_par + "\n\n**Table 4.** Accuracy and bias-corrected areas from the independent stratified "
                 "random sample (Olofsson et al. 2014); +/- = 95% CI half-width.\n\n"
                 + md_table(t4.set_index("class")[["UA", "UA_95CI", "PA", "PA_95CI", "mapped_area_ha", "adjusted_area_ha", "area_95CI_ha"]], fmt=".2f") + "\n")
draft = "\n".join(parts)
display(Markdown(draft))

# Results (auto-generated draft, Project 1)

## 4.1 Model comparison (RQ1)
Under nested spatial cross-validation, XGBoost achieved the highest macro F1 (0.919, 95% CI 0.893-0.938), followed by LightGBM (0.917, 0.893-0.939) (Table 2).

**Table 2.** Nested spatial CV (5 outer folds of 2 km blocks): OA as mean and s.d. over folds; macro F1 from pooled out-of-fold predictions with bootstrap 95% CI.

|  | OA (mean) | OA (s.d.) | macro F1 | macro F1 CI low | macro F1 CI high |
|---|---|---|---|---|---|
| XGBoost | 0.918 | 0.013 | 0.919 | 0.893 | 0.938 |
| LightGBM | 0.916 | 0.027 | 0.917 | 0.893 | 0.939 |
| RF | 0.906 | 0.014 | 0.907 | 0.882 | 0.929 |
| SVM | 0.866 | 0.046 | 0.865 | 0.837 | 0.891 |
| LR | 0.865 | 0.039 | 0.865 | 0.836 | 0.890 |
| MLP | 0.859 | 0.079 | 0.858 | 0.828 | 0.885 |

## 4.2 Contribution of data sources (RQ2)
Macro F1 rose from 0.791 with monsoon Sentinel-2 bands alone to 0.910 with all feature groups (Table 3).

**Table 3.** Cumulative feature-group ablation (RF, 3 x 5 repeated spatial CV).

| feature set | n features | macro F1 (pooled) | OA (mean) | OA (s.d.) | OA diff vs full | p vs full (corrected t) |
|---|---|---|---|---|---|---|
| S2 monsoon (baseline) | 10.000 | 0.791 | 0.786 | 0.048 | -0.123 | 0.000 |
| + S2 other seasons | 40.000 | 0.862 | 0.859 | 0.044 | -0.050 | 0.005 |
| + indices & temporal | 66.000 | 0.873 | 0.870 | 0.042 | -0.038 | 0.009 |
| + Sentinel-1 | 72.000 | 0.907 | 0.905 | 0.038 | -0.004 | 0.681 |
| + texture | 77.000 | 0.901 | 0.899 | 0.036 | -0.010 | 0.193 |
| + terrain (all) | 81.000 | 0.910 | 0.909 | 0.031 | 0.000 |  |

## 4.3 Map accuracy and class areas (RQ3)
The XGBoost map (Fig. 6) had an overall accuracy of 0.952 (95% CI +/- 0.012), estimated from an independent stratified random sample of 1275 points (Table 4). User's accuracy was lowest for water (0.79 +/- 0.09) and producer's accuracy was lowest for forest (0.95 +/- 0.02). Disagreement was mostly allocation (0.042) rather than quantity (0.006). Shrub grass was the largest class (4,424 +/- 119 ha). Bias correction changed class areas by 0.5% to 21.3% relative to pixel counting, the largest change being for water (-21.3%). 99.7% of the study area was inside the area of applicability.

**Table 4.** Accuracy and bias-corrected areas from the independent stratified random sample (Olofsson et al. 2014); +/- = 95% CI half-width.

| class | UA | UA_95CI | PA | PA_95CI | mapped_area_ha | adjusted_area_ha | area_95CI_ha |
|---|---|---|---|---|---|---|---|
| water | 0.79 | 0.09 | 1.00 | 0.00 | 116.52 | 91.66 | 10.88 |
| forest | 0.95 | 0.02 | 0.95 | 0.02 | 3342.00 | 3368.79 | 103.61 |
| cropland | 0.97 | 0.03 | 0.95 | 0.03 | 1521.80 | 1557.24 | 62.87 |
| built_up | 0.97 | 0.04 | 1.00 | 0.00 | 271.24 | 264.01 | 9.96 |
| mining_bare | 0.95 | 0.05 | 0.97 | 0.03 | 302.36 | 294.51 | 17.80 |
| shrub_grass | 0.95 | 0.02 | 0.95 | 0.02 | 4446.08 | 4423.79 | 119.01 |


## **3. The Methods Paragraph Reviewers Want to See**
State the **validation design** precisely, because that is where most RS-ML papers are attacked. Generate it from the configuration too:

In [3]:
cfg = json.loads((P1 / "config.json").read_text()) if (P1 / "config.json").exists() else {}
methods = f"""**2.5 Model development and validation.** Reference data (n = {meta['n_train'] if meta else 'N'}) were split into spatial blocks of
{cfg.get('cv', {}).get('block_size_m', 2000) / 1000:.0f} km, and all model development (feature selection, hyperparameter tuning) was performed in nested
spatial cross-validation (5 outer and 3 inner folds of blocks) to avoid optimistic bias from spatial autocorrelation (Ploton et al., 2020).
Six classifiers were compared with an equal tuning budget; differences were tested with McNemar's test with Holm correction.
The final model ({meta['model'] if meta else 'model'}) was refit on all reference data.
**2.6 Accuracy assessment and area estimation.** Map accuracy was assessed with an independent stratified random sample (strata = map classes;
sample size from Olofsson et al. (2014, Eq. 13) for a target standard error of overall accuracy of {cfg.get('validation', {}).get('target_se_oa', 0.01)}, with at least
{cfg.get('validation', {}).get('n_rare_class', 75)} samples per rare class). Reference labels were interpreted blind to the map. We report the error matrix in
area proportions, overall, user's and producer's accuracies, and bias-corrected class areas with 95% confidence intervals, together with
quantity and allocation disagreement (Pontius & Millones, 2011)."""
print(textwrap.fill(methods.replace("\n", " "), 140))

**2.5 Model development and validation.** Reference data (n = 598) were split into spatial blocks of 2 km, and all model development
(feature selection, hyperparameter tuning) was performed in nested spatial cross-validation (5 outer and 3 inner folds of blocks) to avoid
optimistic bias from spatial autocorrelation (Ploton et al., 2020). Six classifiers were compared with an equal tuning budget; differences
were tested with McNemar's test with Holm correction. The final model (XGBoost) was refit on all reference data. **2.6 Accuracy assessment
and area estimation.** Map accuracy was assessed with an independent stratified random sample (strata = map classes; sample size from
Olofsson et al. (2014, Eq. 13) for a target standard error of overall accuracy of 0.01, with at least 75 samples per rare class). Reference
labels were interpreted blind to the map. We report the error matrix in area proportions, overall, user's and producer's accuracies, and
bias-corrected class areas with 95% con

## **4. Title, Abstract, Keywords, Highlights**
- **Title**: specific and informative. "Multi-season Sentinel-1/2 random forest mapping of land cover in the Damodar coalfield, India, with design-based area estimates" beats "A novel deep learning framework for LULC classification".
- **Abstract**: a structured story with **numbers and uncertainty**.
- **Keywords**: do not repeat title words; add the sensor, the method family, the region and the application.
- **Highlights** (Elsevier journals): 3-5 bullets of at most 85 characters each, including spaces.

In [4]:
def check_highlights(hl, max_chars=85):
    for h in hl: print(f"[{'OK ' if len(h) <= max_chars else 'TOO LONG'}] ({len(h):>3}) {h}")

highlights = ["Multi-season Sentinel-1/2 features improve land-cover mapping in a mining landscape",
              "Models were tuned and compared with nested spatial cross-validation",
              "Class areas were bias-corrected with an independent stratified random sample",
              "Mining and built-up areas are reported with 95% confidence intervals",
              "All code and data are archived for full reproducibility (Zenodo DOI)"]
check_highlights(highlights)

oa_val = float(re.search(r"overall accuracy of ([0-9.]+)", res_par).group(1)) if res_par else float("nan")
gain = (t3["macro F1 (pooled)"].iloc[-1] - t3["macro F1 (pooled)"].iloc[0]) * 100 if t3 is not None else float("nan")
best_txt = f"{best} performed best (macro F1 {t2.loc[best, 'macro F1']:.2f}, 95% CI {t2.loc[best, 'macro F1 CI low']:.2f}-{t2.loc[best, 'macro F1 CI high']:.2f})" if t2 is not None else "<model> performed best"
abstract = f"""Coal mining and urban growth are rapidly transforming the Damodar valley, India, but up-to-date land-cover maps with quantified
uncertainty are lacking. We mapped six land-cover classes at 20 m using multi-season Sentinel-2 composites, Sentinel-1 backscatter, texture and terrain
features. Six machine-learning classifiers were compared with nested spatial cross-validation; {best_txt}, and multi-source features added
{gain:.1f} percentage points of macro F1 over single-season optical data. An independent stratified random sample gave an overall map accuracy of
{oa_val:.2f}, with bias-corrected class areas and 95% confidence intervals. The workflow, built on open data and archived code, provides a transferable
template for monitoring mining landscapes with defensible accuracy and area statistics."""
abstract = re.sub(r"\s+", " ", abstract)
print(f"abstract: {len(abstract.split())} words\n"); print(textwrap.fill(abstract, 140))

[OK ] ( 83) Multi-season Sentinel-1/2 features improve land-cover mapping in a mining landscape
[OK ] ( 67) Models were tuned and compared with nested spatial cross-validation
[OK ] ( 76) Class areas were bias-corrected with an independent stratified random sample
[OK ] ( 68) Mining and built-up areas are reported with 95% confidence intervals
[OK ] ( 68) All code and data are archived for full reproducibility (Zenodo DOI)
abstract: 114 words

Coal mining and urban growth are rapidly transforming the Damodar valley, India, but up-to-date land-cover maps with quantified uncertainty
are lacking. We mapped six land-cover classes at 20 m using multi-season Sentinel-2 composites, Sentinel-1 backscatter, texture and terrain
features. Six machine-learning classifiers were compared with nested spatial cross-validation; XGBoost performed best (macro F1 0.92, 95% CI
0.89-0.94), and multi-source features added 12.0 percentage points of macro F1 over single-season optical data. An independent stra

## **5. Choosing a Journal**
Match the paper's **contribution** to the journal's scope: a methodological advance, a new application or region, or a dataset. Check the current scope, open-access options and fees on each journal's website. They change, so do not rely on old lists.

| Journal (publisher) | Typical RS-ML fit |
|---|---|
| *Remote Sensing of Environment* (Elsevier) | Methodological advances with broad significance |
| *ISPRS Journal of Photogrammetry and Remote Sensing* (Elsevier) | Methods, deep learning, photogrammetry |
| *IEEE Transactions on Geoscience and Remote Sensing* | Algorithms, signal processing, SAR |
| *IEEE JSTARS* | Applied methods and applications |
| *International Journal of Applied Earth Observation and Geoinformation* (Elsevier) | Applications with sound methods |
| *Remote Sensing* (MDPI) | Broad applications, fast review |
| *International Journal of Remote Sensing* / *Remote Sensing Letters* (Taylor & Francis) | Applications / short communications |
| *GIScience & Remote Sensing* (Taylor & Francis) | GIS + RS applications |
| *Journal of the Indian Society of Remote Sensing* (Springer) | Indian case studies |
| *Earth System Science Data*, *Scientific Data* | When the main contribution is a **dataset** or map product |

**Beware of predatory journals**: unsolicited invitations, very fast "peer review", or a journal missing from the indexes it claims. Check the journal in Scopus or Web of Science and on thinkchecksubmit.org. **Preprints** (EarthArXiv, ESS Open Archive, arXiv) are accepted by most RS journals; check the journal's policy.

## **6. The Cover Letter**

In [5]:
cover = f"""Dear Editor,

We submit the manuscript "<Title>" for consideration as a research article in <Journal>.

Land-cover maps of mining landscapes are widely produced, but are rarely accompanied by spatially independent validation and
bias-corrected area estimates. Our study (i) compares six machine-learning classifiers under nested spatial cross-validation,
(ii) quantifies the contribution of multi-season Sentinel-1/2, texture and terrain features through an ablation study, and
(iii) reports class areas with 95% confidence intervals from an independent probability sample. The map achieved an overall
accuracy of {oa_val:.2f}, and all code and data are openly archived.

We believe the work fits <Journal>'s scope on <scope phrase from the journal's aims>. The manuscript is original, has not been
published and is not under consideration elsewhere; all authors approved the submission. We have no conflicts of interest.
Suggested reviewers (no conflicts): <Name, affiliation, e-mail, expertise> x 3.

Sincerely,
<Corresponding author>, on behalf of all authors"""
print(cover)

Dear Editor,

We submit the manuscript "<Title>" for consideration as a research article in <Journal>.

Land-cover maps of mining landscapes are widely produced, but are rarely accompanied by spatially independent validation and
bias-corrected area estimates. Our study (i) compares six machine-learning classifiers under nested spatial cross-validation,
(ii) quantifies the contribution of multi-season Sentinel-1/2, texture and terrain features through an ablation study, and
(iii) reports class areas with 95% confidence intervals from an independent probability sample. The map achieved an overall
accuracy of 0.95, and all code and data are openly archived.

We believe the work fits <Journal>'s scope on <scope phrase from the journal's aims>. The manuscript is original, has not been
published and is not under consideration elsewhere; all authors approved the submission. We have no conflicts of interest.
Suggested reviewers (no conflicts): <Name, affiliation, e-mail, expertise> x 3.

Since

## **7. Responding to Reviewers**
Rules: thank the reviewers, answer **every** point, quote the comment, say **what changed and where** (page and line), and stay polite even when we disagree, giving evidence. Make changes visible in a tracked-changes version.

In [6]:
comments = [("R1.1", "The use of random cross-validation likely inflates accuracy due to spatial autocorrelation.",
             "We agree this is critical. All reported CV results already use nested spatial block CV (2 km blocks). We added a comparison with random CV "
             "showing an optimism of X points (new Table S2) and clarified the design in Section 2.5.", "Sec. 2.5, lines 212-230; Table S2"),
            ("R1.2", "Why is kappa not reported?",
             "Following Pontius & Millones (2011) and Foody (2020), kappa is not informative for map accuracy. We report OA, UA and PA with 95% CIs, "
             "and quantity and allocation disagreement, and added a sentence explaining this choice.", "Sec. 2.6, lines 245-248"),
            ("R2.1", "The deep learning model should outperform RF; please tune it better.",
             "We extended the tuning budget of the MLP (now 30 candidates; Table S3). The ranking did not change. With ~600 reference samples, "
             "tree ensembles are competitive, consistent with Maxwell et al. (2018). We discuss this in Section 5.2.", "Sec. 5.2, lines 410-422; Table S3")]
resp = pd.DataFrame(comments, columns=["#", "Reviewer comment", "Response", "Change in manuscript"]).set_index("#")
display(Markdown(md_table(resp)))

| # | Reviewer comment | Response | Change in manuscript |
|---|---|---|---|
| R1.1 | The use of random cross-validation likely inflates accuracy due to spatial autocorrelation. | We agree this is critical. All reported CV results already use nested spatial block CV (2 km blocks). We added a comparison with random CV showing an optimism of X points (new Table S2) and clarified the design in Section 2.5. | Sec. 2.5, lines 212-230; Table S2 |
| R1.2 | Why is kappa not reported? | Following Pontius & Millones (2011) and Foody (2020), kappa is not informative for map accuracy. We report OA, UA and PA with 95% CIs, and quantity and allocation disagreement, and added a sentence explaining this choice. | Sec. 2.6, lines 245-248 |
| R2.1 | The deep learning model should outperform RF; please tune it better. | We extended the tuning budget of the MLP (now 30 candidates; Table S3). The ranking did not change. With ~600 reference samples, tree ensembles are competitive, consistent with Maxwell et al. (2018). We discuss this in Section 5.2. | Sec. 5.2, lines 410-422; Table S3 |

## **8. A Manuscript Linter for RS-ML Papers**
Many weaknesses can be found automatically. The function below scans a draft for typical red flags:
- kappa reported
- accuracies without uncertainty
- "random split" without any spatial validation
- overclaiming words ("novel", "first", "outperforms all")
- missing availability statements

It is not a substitute for thinking, but it catches the obvious before a reviewer does.

In [7]:
RULES = [(r"\bkappa\b", "Kappa reported: replace with OA/UA/PA with CIs and quantity/allocation disagreement (Pontius & Millones 2011)."),
         (r"accuracy of (?:0\.9\d+|9\d(?:\.\d+)?\s?%)(?![^.]*(?:\+/-|±|CI|confidence))", "High accuracy without an uncertainty statement in the same sentence."),
         (r"random(?:ly)? (?:split|selected|divided)(?![^.]*spatial)", "Random split mentioned; is there spatial or independent validation?"),
         (r"\b(?:novel|first time|for the first|outperform(?:s|ed)? all|state-of-the-art)\b", "Overclaiming language: support it with evidence or soften it."),
         (r"\b(?:significant(?:ly)?)\b(?![^.]*(?:p\s?[<=]|test|McNemar|t-test))", "'Significant' without a statistical test in the same sentence."),
         (r"\btrain(?:ing)? (?:and|&) test(?:ing)? (?:sets? )?(?:were|was) (?:taken|drawn) from the same (?:image|scene|polygons?)", "Same polygons/image for training and testing: leakage."),
         (r"\b(?:areas?|hectares?) (?:were |was )?(?:obtained |derived |computed )?(?:from|by) pixel counting\b|\bmapped area\b(?![^.]*adjust)", "Areas from pixel counting: report bias-corrected areas from a probability sample.")]
REQUIRED = {"data availability": r"data availab", "code availability": r"code availab", "spatial validation": r"spatial (?:block|cross|cv)",
            "uncertainty": r"confidence interval|\b95%|±|\+/-"}

def lint(text):
    issues = []
    for sent in re.split(r"(?<=[.!?])\s+", text):
        for pat, msg in RULES:
            if re.search(pat, sent, flags=re.I): issues.append((msg, sent.strip()[:110]))
    missing = [k for k, pat in REQUIRED.items() if not re.search(pat, text, flags=re.I)]
    return issues, missing

bad_draft = """We propose a novel deep learning framework that outperforms all existing methods. Samples were randomly split into 70% training and 30% testing.
The model achieved an overall accuracy of 98.7% and a kappa of 0.97. The improvement was significant. The mapped area of forest was 1234 ha."""
for name, txt in [("weak draft", bad_draft), ("our generated draft", draft + methods + abstract)]:
    issues, missing = lint(txt)
    print(f"=== {name}: {len(issues)} flags; missing sections: {missing or 'none'}")
    for msg, sent in issues[:8]: print(f"  - {msg}\n      \"{sent}\"")

=== weak draft: 6 flags; missing sections: ['data availability', 'code availability', 'spatial validation', 'uncertainty']
  - Overclaiming language: support it with evidence or soften it.
      "We propose a novel deep learning framework that outperforms all existing methods."
  - Random split mentioned; is there spatial or independent validation?
      "Samples were randomly split into 70% training and 30% testing."
  - Kappa reported: replace with OA/UA/PA with CIs and quantity/allocation disagreement (Pontius & Millones 2011).
      "The model achieved an overall accuracy of 98.7% and a kappa of 0.97."
  - High accuracy without an uncertainty statement in the same sentence.
      "The model achieved an overall accuracy of 98.7% and a kappa of 0.97."
  - 'Significant' without a statistical test in the same sentence.
      "The improvement was significant."
  - Areas from pixel counting: report bias-corrected areas from a probability sample.
      "The mapped area of forest was 1234 

The generated draft still lacks data and code availability statements. Those come from Day 99 (`docs/data_availability.md`).

## **9. Ethics and Good Conduct**
- **Authorship**: everyone listed made a substantial contribution. Use the **CRediT** taxonomy (Conceptualization, Methodology, Software, Validation, Formal analysis, Investigation, Data curation, Writing - original draft, Writing - review & editing, Visualization, Supervision, Funding acquisition).
- **AI-assisted writing and coding**: most publishers require authors to **disclose** the use of generative AI tools in the manuscript (often in a dedicated statement). AI tools cannot be authors. We are responsible for every sentence, number and reference. **Verify every citation**: AI tools can produce plausible but non-existent references.
- **Data**: respect licences (Copernicus data are free with attribution; commercial imagery often forbids redistribution). Protect personal and land-owner information in field data.
- **Duplicate publication and salami slicing**: one study, one paper. Reusing our own text requires citation.
- **Sensitive applications**: think about dual use (e.g. mapping vulnerable communities) and communicate uncertainty to decision makers.

## **10. The 100-Day Map**

In [8]:
roadmap = pd.DataFrame([
    ("01 Foundations and tools", "1-8", "Python, NumPy, pandas, visualisation, linear algebra, calculus"),
    ("02 Probability, statistics, optimisation", "9-14", "distributions, inference, MLE/MAP, gradient descent"),
    ("03 Data preparation and features", "15-21", "cleaning, missing data, encoding, scaling, feature engineering, imbalance"),
    ("04 Regression", "22-26", "linear, regularised, polynomial, GLMs, evaluation"),
    ("05 Classification", "27-34", "logistic, kNN, Bayes, SVM, trees, metrics, calibration"),
    ("06 Ensembles", "35-40", "bagging, RF, boosting, XGBoost/LightGBM/CatBoost, stacking"),
    ("07 Model selection, tuning, explainability", "41-45", "CV, nested CV, Bayesian optimisation, SHAP"),
    ("08 Unsupervised learning", "46-52", "clustering, PCA, manifold learning, anomalies, recommenders"),
    ("09 Probabilistic ML, time series, semi-supervised", "53-57", "Bayesian methods, GPs, forecasting, self-training"),
    ("10 Deep learning foundations", "58-66", "MLPs, backprop, CNNs, transfer learning, segmentation, RNNs"),
    ("11 Advanced deep learning", "67-76", "attention, transformers, SSL, VAEs, GANs, diffusion, GNNs, RL, LLMs"),
    ("12 Trustworthy ML and MLOps", "77-85", "uncertainty, fairness, domain shift, PIML, MLOps, deployment, monitoring, scale"),
    ("13 Remote-sensing ML publication projects", "86-100", "RS data, GEE, spatial validation, 7 projects, reproducibility, publishing")],
    columns=["Module", "Days", "Core topics"]).set_index("Module")
display(Markdown(md_table(roadmap)))

| Module | Days | Core topics |
|---|---|---|
| 01 Foundations and tools | 1-8 | Python, NumPy, pandas, visualisation, linear algebra, calculus |
| 02 Probability, statistics, optimisation | 9-14 | distributions, inference, MLE/MAP, gradient descent |
| 03 Data preparation and features | 15-21 | cleaning, missing data, encoding, scaling, feature engineering, imbalance |
| 04 Regression | 22-26 | linear, regularised, polynomial, GLMs, evaluation |
| 05 Classification | 27-34 | logistic, kNN, Bayes, SVM, trees, metrics, calibration |
| 06 Ensembles | 35-40 | bagging, RF, boosting, XGBoost/LightGBM/CatBoost, stacking |
| 07 Model selection, tuning, explainability | 41-45 | CV, nested CV, Bayesian optimisation, SHAP |
| 08 Unsupervised learning | 46-52 | clustering, PCA, manifold learning, anomalies, recommenders |
| 09 Probabilistic ML, time series, semi-supervised | 53-57 | Bayesian methods, GPs, forecasting, self-training |
| 10 Deep learning foundations | 58-66 | MLPs, backprop, CNNs, transfer learning, segmentation, RNNs |
| 11 Advanced deep learning | 67-76 | attention, transformers, SSL, VAEs, GANs, diffusion, GNNs, RL, LLMs |
| 12 Trustworthy ML and MLOps | 77-85 | uncertainty, fairness, domain shift, PIML, MLOps, deployment, monitoring, scale |
| 13 Remote-sensing ML publication projects | 86-100 | RS data, GEE, spatial validation, 7 projects, reproducibility, publishing |

## **11. What Next?**
1. **Do Project 1 with real data**: run the Day 87 GEE code for our own area and season, collect reference data (field or VHR interpretation), and follow Days 88-91 step by step. Change only `data/study_area`.
2. **Pick one project as a paper**: the crop-type (Days 92-93), flood (94), landslide (95), biomass (96) or change (98) template, combined with our thesis data.
3. **Follow the field**: read RSE, ISPRS and IEEE TGRS tables of contents monthly; explore TorchGeo, geospatial foundation models (Prithvi, SatMAE, Clay) and benchmark datasets (BigEarthNet, Sen1Floods11, EuroCrops).
4. **Share**: publish our code with a DOI, write a preprint, present at a conference (IGARSS, ISPRS, the Indian Society of Remote Sensing symposia).

## **Practice Problems**
1. Write a 250-word structured abstract for **Project 5 (biomass)** using the numbers in `project5_biomass/table3_totals.csv` and `table1_models.csv`. Generate it programmatically.
2. Write 5 highlights (max 85 characters each) for **Project 3 (floods)** and check them with `check_highlights`.
3. A reviewer writes: *"Your test set is too small (n = 50 per class)."* Draft a response using the CIs from Table 4.
4. *(Advanced)* Extend `lint()` with a rule that flags accuracy numbers given with more than one decimal place in percent (false precision), and a rule for missing sensor dates.

In [9]:
# Solution 1
P5 = Path("project5_biomass"); t5 = load_csv(P5 / "table3_totals.csv", index_col=0); m5 = load_csv(P5 / "table1_models.csv")
iv5 = load_csv(P5 / "table2_intervals.csv", index_col=0)
cov5 = float(iv5.loc["CQR (LightGBM)", "coverage (target 0.90)"]) if iv5 is not None else float("nan")
if t5 is not None and m5 is not None:
    best5 = m5.sort_values("RMSE").iloc[0]; g = t5.loc["model-assisted GREG"]; d = t5.loc["design-based (plots only)"]
    ab5 = (f"Forest biomass estimates with defensible uncertainty are needed for carbon accounting in eastern India. We combined 250 field plots with "
           f"Sentinel-1/2 and canopy-height data to map above-ground biomass (AGB) at 20 m. Under spatial cross-validation, the best model "
           f"({best5['model']}, {best5['features']}) achieved RMSE = {best5['RMSE']:.1f} Mg/ha (rRMSE {best5['rRMSE %']:.0f}%, R2 = {best5['R2']:.2f}), with the "
           f"saturation-related under-prediction typical of high-biomass stands. Conformalised quantile regression gave pixel-level 90% prediction "
           f"intervals with an achieved coverage of {cov5:.2f} for a 0.90 target under spatial cross-validation. A model-assisted (GREG) estimator gave a total of {g['total AGB (Mg)'] / 1000:.1f} +/- "
           f"{g['95% CI half-width (Mg)'] / 1000:.1f} Gg AGB ({g['total carbon (Mg C)'] / 1000:.1f} Gg C), {(d['95% CI half-width (Mg)'] / g['95% CI half-width (Mg)']) ** 2:.0f} "
           f"times more efficient than the plot-only estimate. The approach supports transparent, uncertainty-aware forest carbon reporting.")
    print(f"{len(ab5.split())} words\n" + textwrap.fill(ab5, 140))

112 words
Forest biomass estimates with defensible uncertainty are needed for carbon accounting in eastern India. We combined 250 field plots with
Sentinel-1/2 and canopy-height data to map above-ground biomass (AGB) at 20 m. Under spatial cross-validation, the best model (random
forest, S1+S2+CHM) achieved RMSE = 21.2 Mg/ha (rRMSE 38%, R2 = 0.85), with the saturation-related under-prediction typical of high-biomass
stands. Conformalised quantile regression gave pixel-level 90% prediction intervals with an achieved coverage of 0.81 for a 0.90 target
under spatial cross-validation. A model-assisted (GREG) estimator gave a total of 145.0 +/- 6.7 Gg AGB (68.1 Gg C), 6 times more efficient
than the plot-only estimate. The approach supports transparent, uncertainty-aware forest carbon reporting.


In [10]:
# Solution 2
check_highlights(["Change detection with HAND masks gives precise but incomplete flood maps",
                  "A compact U-Net recovers flooded vegetation missed by thresholds",
                  "Errors were analysed by condition: wind, vegetation, urban, radar shadow",
                  "SAR augmentation must respect the radar look direction",
                  "Flood exposure of cropland and settlements is reported per event"])

[OK ] ( 72) Change detection with HAND masks gives precise but incomplete flood maps
[OK ] ( 64) A compact U-Net recovers flooded vegetation missed by thresholds
[OK ] ( 72) Errors were analysed by condition: wind, vegetation, urban, radar shadow
[OK ] ( 54) SAR augmentation must respect the radar look direction
[OK ] ( 64) Flood exposure of cropland and settlements is reported per event


In [11]:
# Solution 3
if t4 is not None:
    w = t4["UA_95CI"].max(); c = t4.loc[t4["UA_95CI"].idxmax(), "class"]
    print(f"Response: We thank the reviewer. The sample size followed Olofsson et al. (2014, Eq. 13) for a target SE of overall accuracy of 0.01, "
          f"with at least 75 samples per rare class. The resulting 95% CIs are reported for all estimates (Table 4); the widest user's-accuracy CI "
          f"is +/- {w:.3f} ({c}), and overall accuracy is estimated to +/- {float(re.search(r'CI \+/- ([0-9.]+)', res_par).group(1)) if res_par else float('nan'):.3f}. We added a sentence on the "
          f"sample-size rationale (Section 2.6).")

Response: We thank the reviewer. The sample size followed Olofsson et al. (2014, Eq. 13) for a target SE of overall accuracy of 0.01, with at least 75 samples per rare class. The resulting 95% CIs are reported for all estimates (Table 4); the widest user's-accuracy CI is +/- 0.093 (water), and overall accuracy is estimated to +/- 0.012. We added a sentence on the sample-size rationale (Section 2.6).


In [12]:
# Solution 4
RULES.append((r"\b\d{2}\.\d{2,}\s?%", "False precision: report accuracies to one decimal place in percent (or 2-3 decimals as proportions)."))
RULES.append((r"Sentinel-?[12](?![^.]*(?:20\d\d|acquired|dates?))", "Sensor mentioned without acquisition dates in the same sentence."))
issues, _ = lint("The Sentinel-2 images were classified with an accuracy of 91.237% and the OA was 0.91 +/- 0.02.")
for msg, sent in issues: print("-", msg)

- High accuracy without an uncertainty statement in the same sentence.
- False precision: report accuracies to one decimal place in percent (or 2-3 decimals as proportions).
- Sensor mentioned without acquisition dates in the same sentence.


## **Course Wrap-Up**
Over 100 days we went from Python and NumPy, through statistics, classical ML, ensembles, unsupervised and probabilistic learning, deep learning and modern architectures, and trustworthy ML and MLOps, to seven complete remote-sensing studies built to publication standard: LULC, crop type, floods, landslides, biomass, scene classification and change projection. The habits matter more than any single algorithm: **spatially honest validation, quantified uncertainty, reproducible code, and clear reporting**. Use them in our next paper.

## **Key References**
- Day, R. A., & Gastel, B. (2016). *How to Write and Publish a Scientific Paper* (8th ed.). Cambridge University Press.
- Mensh, B., & Kording, K. (2017). Ten simple rules for structuring papers. *PLOS Computational Biology*, 13(9), e1005619.
- Foody, G. M. (2020). Explaining the unsuitability of the kappa coefficient in the assessment and comparison of the accuracy of thematic maps obtained by image classification. *Remote Sensing of Environment*, 239, 111630.
- Olofsson, P. et al. (2014). Good practices for estimating area and assessing accuracy of land change. *Remote Sensing of Environment*, 148, 42-57.
- Maxwell, A. E., Warner, T. A., & Fang, F. (2018). Implementation of machine-learning classification in remote sensing: an applied review. *International Journal of Remote Sensing*, 39(9), 2784-2817.
- Brand, A., Allen, L., Altman, M., Hlava, M., & Scott, J. (2015). Beyond authorship: attribution, contribution, collaboration, and credit. *Learned Publishing*, 28(2), 151-155 (CRediT).
- Committee on Publication Ethics (COPE). Guidance on authorship and AI tools: publicationethics.org.